In [2]:
# Check libraries are available
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("All libraries ready")

All libraries ready


In [3]:
# Download the dataset directly
!wget -q https://raw.githubusercontent.com/several27/FakeNewsCorpus/master/news_sample.csv

# Load it
df = pd.read_csv('news_sample.csv')
print(df.shape)
print(df.head())

(250, 16)
   Unnamed: 0   id                domain        type  \
0           0  141               awm.com  unreliable   
1           1  256     beforeitsnews.com        fake   
2           2  700           cnnnext.com  unreliable   
3           3  768               awm.com  unreliable   
4           4  791  bipartisanreport.com   clickbait   

                                                 url  \
0  http://awm.com/church-congregation-brings-gift...   
1  http://beforeitsnews.com/awakening-start-here/...   
2  http://www.cnnnext.com/video/18526/never-hike-...   
3  http://awm.com/elusive-alien-of-the-sea-caught...   
4  http://bipartisanreport.com/2018/01/21/trumps-...   

                                             content  \
0  Sometimes the power of Christmas will make you...   
1  AWAKENING OF 12 STRANDS of DNA – “Reconnecting...   
2  Never Hike Alone: A Friday the 13th Fan Film U...   
3  When a rare shark was caught, scientists were ...   
4  Donald Trump has the unnerving ab

In [4]:
# See what types of news we have
print(df['type'].value_counts())
print("\nMissing values in content:", df['content'].isnull().sum())

type
fake          155
conspiracy     31
political      23
unreliable      6
junksci         6
bias            6
unknown         6
reliable        3
clickbait       1
hate            1
Name: count, dtype: int64

Missing values in content: 0


In [5]:
# Simplify labels → only FAKE or REAL
def label_news(t):
    if t == 'reliable':
        return 'real'
    else:
        return 'fake'

df['label'] = df['type'].apply(label_news)

print(df['label'].value_counts())
print("\nSample content:")
print(df['content'][0][:200])  # first 200 chars of first article

label
fake    247
real      3
Name: count, dtype: int64

Sample content:
Sometimes the power of Christmas will make you do wild and wonderful things. You do not need to believe in the Holy Trinity to believe in the positive power of doing good for others. The simple act of


In [6]:
# Delete old dataset, get a proper balanced one
import os
os.remove('news_sample.csv')

# This is the standard fake news dataset used in research
!wget -q https://raw.githubusercontent.com/lutzhamel/fake-news/master/data/fake_or_real_news.csv

df = pd.read_csv('fake_or_real_news.csv')
print(df.shape)
print(df.columns.tolist())
print(df['label'].value_counts())

(6335, 4)
['id', 'title', 'text', 'label']
label
REAL    3171
FAKE    3164
Name: count, dtype: int64


In [7]:
# Keep only what we need
df = df[['title', 'text', 'label']]

# Combine title + text for better accuracy
df['content'] = df['title'] + ' ' + df['text']

# Check for missing values
print("Missing values:")
print(df.isnull().sum())
print("\nShape:", df.shape)
print("\nSample content (first 300 chars):")
print(df['content'][0][:300])

Missing values:
title      0
text       0
label      0
content    0
dtype: int64

Shape: (6335, 4)

Sample content (first 300 chars):
You Can Smell Hillary’s Fear Daniel Greenfield, a Shillman Journalism Fellow at the Freedom Center, is a New York writer focusing on radical Islam. 
In the final stretch of the election, Hillary Rodham Clinton has gone to war with the FBI. 
The word “unprecedented” has been thrown around so often th


In [8]:
# Convert labels to numbers (FAKE=0, REAL=1)
df['label_num'] = df['label'].map({'FAKE': 0, 'REAL': 1})

# Split into input (X) and output (y)
X = df['content']
y = df['label_num']

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nFake in training:", (y_train==0).sum())
print("Real in training:", (y_train==1).sum())

Training samples: 5068
Testing samples: 1267

Fake in training: 2536
Real in training: 2532


In [9]:
# Convert text to numbers using TF-IDF
# (machine can't read words, only numbers)
tfidf = TfidfVectorizer(
    max_features=5000,  # use top 5000 words only
    stop_words='english',  # ignore common words like "the", "is"
    ngram_range=(1,2)  # consider single words AND pairs of words
)

# Learn vocabulary from training data and transform it
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data using SAME vocabulary
X_test_tfidf = tfidf.transform(X_test)

print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)
print("\nVocabulary size:", len(tfidf.vocabulary_))

Training matrix shape: (5068, 5000)
Testing matrix shape: (1267, 5000)

Vocabulary size: 5000


In [10]:
# Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Test it
y_pred = model.predict(X_test_tfidf)

# See how good it is
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=['FAKE', 'REAL']))

Accuracy: 0.914759273875296

Detailed Report:
              precision    recall  f1-score   support

        FAKE       0.91      0.92      0.91       628
        REAL       0.92      0.91      0.91       639

    accuracy                           0.91      1267
   macro avg       0.91      0.91      0.91      1267
weighted avg       0.91      0.91      0.91      1267



In [11]:
# Function to predict any news
def predict_news(text):
    # Convert text to numbers using same tfidf
    text_tfidf = tfidf.transform([text])

    # Predict
    prediction = model.predict(text_tfidf)[0]
    confidence = model.predict_proba(text_tfidf)[0]

    label = "REAL ✅" if prediction == 1 else "FAKE ❌"
    conf_score = confidence[prediction] * 100

    print(f"Prediction: {label}")
    print(f"Confidence: {conf_score:.1f}%")
    print("-"*40)

# Test with examples
print("TEST 1:")
predict_news("Scientists discover new vaccine that prevents cancer with 95% success rate in trials")

print("TEST 2:")
predict_news("BREAKING: Obama secretly confessed to being born in Kenya, deep state coverup revealed")

print("TEST 3:")
predict_news("Federal Reserve raises interest rates by 0.25% amid inflation concerns")

TEST 1:
Prediction: FAKE ❌
Confidence: 71.7%
----------------------------------------
TEST 2:
Prediction: FAKE ❌
Confidence: 74.7%
----------------------------------------
TEST 3:
Prediction: FAKE ❌
Confidence: 62.5%
----------------------------------------


In [12]:
# Check what the model thinks "real" news looks like
print("Testing with election-style text:")
predict_news("President Trump signed executive order on immigration policy today in Washington")

print("Testing with another political one:")
predict_news("Hillary Clinton addressed supporters in New York following the election results")

print("Testing with original dataset sample:")
sample_real = df[df['label']=='REAL']['content'].iloc[0][:300]
print("Sample text:", sample_real[:100])
predict_news(sample_real)

Testing with election-style text:
Prediction: FAKE ❌
Confidence: 61.0%
----------------------------------------
Testing with another political one:
Prediction: FAKE ❌
Confidence: 88.3%
----------------------------------------
Testing with original dataset sample:
Sample text: Kerry to go to Paris in gesture of sympathy U.S. Secretary of State John F. Kerry said Monday that h
Prediction: REAL ✅
Confidence: 92.2%
----------------------------------------


In [13]:
# Test with more samples directly from dataset
print("=== REAL news samples from dataset ===")
real_samples = df[df['label']=='REAL']['content'].sample(3, random_state=42)
for i, text in enumerate(real_samples):
    print(f"\nSample {i+1} (first 100 chars): {text[:100]}")
    predict_news(text)

print("\n=== FAKE news samples from dataset ===")
fake_samples = df[df['label']=='FAKE']['content'].sample(3, random_state=42)
for i, text in enumerate(fake_samples):
    print(f"\nSample {i+1} (first 100 chars): {text[:100]}")
    predict_news(text)

=== REAL news samples from dataset ===

Sample 1 (first 100 chars): Going Back to the Future in 2016? (CNN) Who among the nascent field of 2016 contenders represents th
Prediction: REAL ✅
Confidence: 96.1%
----------------------------------------

Sample 2 (first 100 chars): Dem insiders: Sanders failed to dent Clinton Killing Obama administration rules, dismantling Obamaca
Prediction: REAL ✅
Confidence: 94.4%
----------------------------------------

Sample 3 (first 100 chars): Fuming over Ryan, some conservative voices turn on the Freedom Caucus Rep. Mark Meadows (R-N.C.) has
Prediction: REAL ✅
Confidence: 95.0%
----------------------------------------

=== FAKE news samples from dataset ===

Sample 1 (first 100 chars): The Retirement Nightmare: “There Will Be Life Altering Ramifications For Those Who Can’t Or Won’t Ad
Prediction: FAKE ❌
Confidence: 92.0%
----------------------------------------

Sample 2 (first 100 chars): The Left Turns on Bob Dylan for His Pro-Israel Views, Refusa

In [14]:
import pickle

# Save model and tfidf vectorizer
with open('fake_news_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("Model saved ✅")
print("Vectorizer saved ✅")

# Save to Google Drive so you don't lose it
from google.colab import drive
drive.mount('/content/drive')

Model saved ✅
Vectorizer saved ✅
Mounted at /content/drive


In [15]:
import shutil
import os

# Create project folder in Drive
os.makedirs('/content/drive/MyDrive/FakeNewsDetector', exist_ok=True)

# Copy files there
shutil.copy('fake_news_model.pkl', '/content/drive/MyDrive/FakeNewsDetector/')
shutil.copy('tfidf_vectorizer.pkl', '/content/drive/MyDrive/FakeNewsDetector/')
shutil.copy('fake_or_real_news.csv', '/content/drive/MyDrive/FakeNewsDetector/')

print("All files saved to Google Drive ✅")

All files saved to Google Drive ✅
